# 📐 03 - Shape and Outliers

Notebook 02 showed that Titanic fares have a mean more than twice their median, and a 99th percentile far above the 90th. Those are symptoms. This notebook names the cause (**skew**), measures it, and then hunts down the individual values responsible (**outliers**).

The big idea: an outlier is a value that's **unusual**. It is *not* automatically a value that's **wrong**. Finding outliers is easy. Deciding what to do with them takes thought.

## ✅ Learning Goals
- Describe a distribution's shape: symmetric, right-skewed, or left-skewed
- Measure skew with `.skew()` and tail heaviness with `.kurt()`, and read rough thresholds
- Flag outliers with the **1.5 × IQR rule** and with **z-scores**, and explain how the two differ
- Inspect flagged rows before deciding to keep, fix, cap, or remove them
- Use a log transform to tame a right-skewed column

In [ ]:
import numpy as np
import pandas as pd

titanic = pd.read_csv('titanic.csv')
fare = titanic['Fare']
age = titanic['Age']

## 1. Shape: Which Way Does the Tail Go?
Picture sorting all the values into bins and stacking them up (Module 08 will draw this as a histogram).

| Shape | What it looks like | Mean vs. median | Real examples |
|---|---|---|---|
| **Symmetric** | Both sides mirror each other | about equal | heights, exam scores |
| **Right-skewed** (positive) | Most values low, a long tail of large values | mean **>** median | incomes, prices, fares |
| **Left-skewed** (negative) | Most values high, a long tail of small values | mean **<** median | age at retirement, easy-test scores |

The tail drags the mean toward it. That's why the mean-vs-median gap from notebook 02 is a quick skew detector.

## 2. Measuring Skew and Kurtosis
`.skew()` puts a number on the lopsidedness. A rough guide:

| `.skew()` | Reading |
|---|---|
| between −0.5 and 0.5 | roughly symmetric |
| 0.5 to 1 (or −0.5 to −1) | moderately skewed |
| beyond 1 (or −1) | strongly skewed |

`.kurt()` measures how **heavy the tails** are compared with a bell curve (pandas reports *excess* kurtosis, so a bell curve is about 0). Large positive kurtosis means extreme values show up more often than a bell curve would predict.

In [ ]:
shape = pd.DataFrame({
    'mean': titanic[['Age', 'Fare', 'SibSp']].mean(),
    'median': titanic[['Age', 'Fare', 'SibSp']].median(),
    'skew': titanic[['Age', 'Fare', 'SibSp']].skew(),
    'kurtosis': titanic[['Age', 'Fare', 'SibSp']].kurt(),
}).round(2)
shape

- `Age` is nearly symmetric (skew ≈ 0.4), and its mean and median are close.
- `Fare` is **strongly** right-skewed (skew ≈ 4.8) with very heavy tails (kurtosis ≈ 33). A few tickets are wildly expensive.
- `SibSp` is right-skewed too. Most passengers traveled with 0 or 1 sibling/spouse, and a few large families stretch the tail.

> 📝 These are guidelines, not laws. Always pair the number with a look at the data, and in Module 08, a plot.

## 3. Finding Outliers: The 1.5 × IQR Rule
The most common rule (and the one box plots use, as you'll see in Module 08) builds two **fences** from the quartiles:

```text
lower fence = Q1 − 1.5 × IQR
upper fence = Q3 + 1.5 × IQR
```

Anything outside the fences is flagged. Because it's built from quartiles, the rule itself isn't distorted by the outliers it's looking for.

This is worth a function, since you'll reuse it (Module 03b: write a function once you'd otherwise repeat yourself):

In [ ]:
def iqr_fences(series, k=1.5):
    """Return the (lower, upper) outlier fences for a numeric Series."""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


low, high = iqr_fences(fare)
print(f'Fare fences: {low:.2f} to {high:.2f}')

fare_outlier = (fare < low) | (fare > high)
print(f'Flagged: {fare_outlier.sum()} of {len(fare)} fares ({fare_outlier.mean():.1%})')

The lower fence is negative, so no fare can fall below it. Every flagged fare is on the high side, which is exactly what a right-skewed column predicts.

## 4. Look Before You Decide
Never act on a flag without looking at the rows (Module 06's "inspect first"). Use the mask to pull them out:

In [ ]:
flagged = titanic[fare_outlier]
flagged['Pclass'].value_counts()

In [ ]:
flagged.nlargest(5, 'Fare')[['Name', 'Pclass', 'Fare', 'Ticket', 'Cabin']]

The flagged fares are overwhelmingly **1st class**, and the most expensive tickets are shared by people traveling together. These aren't typos. They're real, expensive tickets.

Now compare with how the rule behaves on `Age`:

In [ ]:
low, high = iqr_fences(age)
print(f'Age fences: {low:.1f} to {high:.1f}')
titanic.loc[(age < low) | (age > high), ['Name', 'Age', 'Pclass']].sort_values('Age')

A handful of passengers aged 65 to 80. Again: unusual, not wrong.

### What to Do with an Outlier

| The outlier is… | Example | Action |
|---|---|---|
| **Impossible** | age 250, a negative price | It's an error. Fix it if you can, otherwise make it `NaN` (Module 06). |
| **A placeholder** | `-1`, `999` | Replace with `NaN` (Module 06). |
| **Real but extreme** | a £512 ticket, an 80-year-old | **Keep it.** Use robust statistics (median, IQR), or report it separately. |
| **Real, but it swamps a model or chart** | a few huge values hiding everyone else | Consider **capping** or a **log transform** (section 6), and say that you did. |

> ⚠️ Deleting real outliers because they're inconvenient changes the answer to the question. If the question is "what did a typical passenger pay?", report the median. Don't delete the rich people.

## 5. A Second Method: Z-Scores
A **z-score** says how many standard deviations a value is from the mean:

```text
z = (value − mean) / std
```

A common rule flags values with |z| > 3.

In [ ]:
fare_z = (fare - fare.mean()) / fare.std()

z_outlier = fare_z.abs() > 3
print('Flagged by z-score:  ', z_outlier.sum())
print('Flagged by IQR rule: ', fare_outlier.sum())

Why so different? The z-score is built from the **mean and std**, and notebook 02 showed both are inflated by outliers. On a skewed column, the outliers stretch the yardstick that's measuring them, so fewer values look extreme.

| Method | Built from | Works best on |
|---|---|---|
| 1.5 × IQR | quartiles (robust) | any shape, especially skewed data |
| \|z\| > 3 | mean and std (not robust) | roughly symmetric, bell-shaped data |

✅ **Your Turn:** Compute z-scores for `age` (it's close to symmetric). Do the z-score rule and the IQR rule agree better on `age` than on `fare`?

In [ ]:
# Your turn here

## 6. Taming Skew with a Log Transform
For right-skewed, positive data (money, counts, sizes), taking the **logarithm** squeezes the long tail and spreads out the crowded low end. `np.log1p(x)` computes log(1 + x), which safely handles zeros.

In [ ]:
log_fare = np.log1p(fare)
print(f'Skew before: {fare.skew():.2f}')
print(f'Skew after:  {log_fare.skew():.2f}')

low, high = iqr_fences(log_fare)
print('IQR outliers after log:', ((log_fare < low) | (log_fare > high)).sum())

After the log, fare is close to symmetric and far fewer values are flagged. Nothing was deleted: the same passengers are all there, measured on a different scale. In Module 08 you'll use the same idea as a **log-scale axis** on a chart.

## 7. Capping (Winsorizing)
Another option is to **cap** extreme values at a percentile with `.clip()`. Values beyond the cap are set *to* the cap, not removed:

In [ ]:
cap = fare.quantile(0.99)
fare_capped = fare.clip(upper=cap)

pd.DataFrame({'original': fare.describe(), 'capped at 99th pct': fare_capped.describe()}).round(2)

The median and quartiles are unchanged, while the max, mean, and std come down. Capping is a modeling convenience. It's not the truth about the data, so always keep the original column.

---
### 📝 Summary
- Right-skewed: long tail of large values, mean > median. Left-skewed: the reverse.
- `.skew()` beyond ±1 is strong skew. `.kurt()` well above 0 means heavy tails.
- The 1.5 × IQR rule is robust and works on skewed data. The |z| > 3 rule assumes a roughly bell-shaped column.
- An outlier is **unusual**, not necessarily **wrong**. Inspect the rows, then fix errors, and keep real values.
- A log transform or capping can tame skew without deleting anyone. Say that you did it.

## 🏋️ Practice

### Practice 1 — Skew Report (easy)
Load `pokemon.csv` into `poke`. Compute `.skew()` for `HP`, `Attack`, `Defense`, `Sp. Atk`, `Sp. Def`, and `Speed`, sorted from most to least skewed. Which is the most lopsided, and in which direction?

In [ ]:
poke = pd.read_csv('pokemon.csv')
# Your code here

### Practice 2 — Who Are the Outliers? (medium)
Use `iqr_fences()` to flag outliers in `poke['HP']`. How many are there? Show the `Name`, `HP`, and `Legendary` columns for the 10 highest. Are the outliers mostly Legendary?

In [ ]:
# Your code here

### Practice 3 — Outliers Depend on the Group (harder)
A fare that's extreme for a 3rd-class passenger might be normal in 1st class. For **each** `Pclass`, compute the IQR fences on just that class's fares and count the outliers (hint: a loop over `[1, 2, 3]`, filtering each time). Compare the total with the overall count from section 3. In a Markdown cell, explain why the counts differ.

In [ ]:
# Your code here

*Your explanation here.*

## ✏️ Your Turn
Change the `k` argument of `iqr_fences()` to `3` (a common "extreme outliers only" setting) and re-count the `Fare` outliers. Then write one sentence on how you'd choose `k` for a real project.

In [ ]:
# Your turn here

*Your sentence here.*

## 🔥 Challenge (Optional) — Outlier Flags with `groupby().transform()`
Practice 3 used a loop. Pandas can compute a per-group statistic and line it up with the original rows using `.transform()`:
```python
q1 = titanic.groupby('Pclass')['Fare'].transform(lambda s: s.quantile(0.25))
```
Build `q3` the same way, then a boolean column `fare_outlier_in_class` with no loop. Check that its count matches your Practice 3 total.

In [ ]:
# Your code here

---
✅ Next up → [Comparing Groups](04-Comparing_Groups.ipynb): the same statistics, computed for every group at once, and how to compare groups fairly.